 # Section 1: Setup, Unified Paths & Global Settings

 This cell loads essential utilities, configures global Pandas display parameters,

 silences silent-downcasting warnings, and dynamically maps project directories.

In [ ]:
%load_ext autoreload
%autoreload 2

import ast
from pathlib import Path

import numpy as np
import pandas as pd

from core.contracts import EngineInput, FilterPack
from core.paths import (
    GLOBAL_DATA_DIR,
    GLOBAL_PROCESSED_DIR,
    LOCAL_DATA_DIR,
    NOTEBOOKS_RLVR_ROOT,
    OUTPUT_DIR,
    PROJECT_ROOT,
)
from core.settings import TradingConfig
from core.utils import SystemUtils as SU
from data_pipeline import generate_features
from walk_forward import (
    AlphaEngine,
    create_walk_forward_analyzer,
    run_headless_simulation,
)

# 1. Global Configuration Options
config = TradingConfig()

# 2. Silence downcasting warnings & configure wide DataFrame display options
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 3000)

# 3. Path Selection Toggle (Set USE_LOCAL_DATA = False to switch to GLOBAL_DATA_DIR)
USE_LOCAL_DATA = True

data_dir = LOCAL_DATA_DIR if USE_LOCAL_DATA else GLOBAL_DATA_DIR
processed_dir = GLOBAL_PROCESSED_DIR
output_dir = OUTPUT_DIR

print(
    f"Paths Resolved:\n"
    f" - Project Root:         {PROJECT_ROOT}\n"
    f" - Notebooks RLVR Root:  {NOTEBOOKS_RLVR_ROOT}\n"
    f" - Global Data Dir:      {GLOBAL_DATA_DIR}\n"
    f" - Global Processed Dir: {GLOBAL_PROCESSED_DIR}\n"
    f" - Local Data Dir:       {LOCAL_DATA_DIR}\n"
    f" - Output Dir:           {OUTPUT_DIR}\n\n"
    f" - Active Data Dir:      {data_dir} ({'LOCAL' if USE_LOCAL_DATA else 'GLOBAL'})\n"
)

# Define Raw File Paths using selected data_dir
source_type = "LOCAL_DATA_DIR" if USE_LOCAL_DATA else "GLOBAL_DATA_DIR"
print(f"\nLoading files from {source_type}: {data_dir}")

ohlcv_filename = "df_ohlcv.parquet" if USE_LOCAL_DATA else "df_OHLCV_stocks_etfs.parquet"
fed_filename = "df_fed.parquet" if USE_LOCAL_DATA else "High_Yield_Spread_T10Y2Y_Spread.parquet"

df_ohlcv_path = data_dir / ohlcv_filename
df_fed_path = data_dir / fed_filename
df_indices_path = data_dir / "df_indices.parquet"


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Paths Resolved:
 - Project Root:         C:\Users\ping\Files_win10\python\py311\stocks
 - Notebooks RLVR Root:  C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
 - Global Data Dir:      C:\Users\ping\Files_win10\python\py311\stocks\data
 - Global Processed Dir: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
 - Local Data Dir:       C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
 - Output Dir:           C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output

 - Active Data Dir:      C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data (LOCAL)


Loading files from LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data


 # Section 2: Ingest Raw and Processed Datasets

 Safely loads raw inputs and previously stored processed matrices with existence verifications.

In [67]:
# # Define Raw File Paths
# df_ohlcv_path = data_dir / "df_OHLCV_stocks_etfs.parquet"
# df_fed_path = data_dir / "High_Yield_Spread_T10Y2Y_Spread.csv"
# df_indices_path = data_dir / "df_indices.parquet"

# Load Raw Files
if df_ohlcv_path.exists():
    df_ohlcv = pd.read_parquet(df_ohlcv_path)
    print("[OK] Successfully loaded df_ohlcv!")
else:
    print(f"[ERROR] File not found: {df_ohlcv_path}")

if df_fed_path.exists():
    # df_fed = pd.read_csv(df_fed_path)
    df_fed = pd.read_parquet(df_fed_path)
    print("[OK] Successfully loaded df_fed!")
else:
    print(f"[ERROR] File not found: {df_fed_path}")

if df_indices_path.exists():
    df_indices = pd.read_parquet(df_indices_path)
    print("[OK] Successfully loaded df_indices!")
else:
    print(f"[ERROR] File not found: {df_indices_path}")

[OK] Successfully loaded df_ohlcv!
[OK] Successfully loaded df_fed!
[OK] Successfully loaded df_indices!


In [68]:
# Define Processed File Paths
features_df_path = data_dir / "features_df.parquet"
macro_df_path = data_dir / "macro_df.parquet"
# features_df_path = current_dir / "data" / "features_df.parquet"
# macro_df_path = current_dir / "data" / "macro_df.parquet"
# df_close_wide_path = processed_dir / "df_close_wide.parquet"
# df_atrp_wide_path = processed_dir / "df_atrp_wide.parquet"
# df_trp_wide_path = processed_dir / "df_trp_wide.parquet"

# Load Processed Files
if features_df_path.exists():
    features_df = pd.read_parquet(features_df_path)
    print(f"[OK] Loaded features_df  | Shape: {features_df.shape}")
else:
    print(f"[ERROR] File not found: {features_df_path}")

if macro_df_path.exists():
    macro_df = pd.read_parquet(macro_df_path)
    print(f"[OK] Loaded macro_df     | Shape: {macro_df.shape}")
else:
    print(f"[ERROR] File not found: {macro_df_path}")

# if df_close_wide_path.exists():
#     df_close_wide = pd.read_parquet(df_close_wide_path)
#     print(f"[OK] Loaded df_close_wide| Shape: {df_close_wide.shape}")
# else:
#     print(f"[ERROR] File not found: {df_close_wide_path}")

# if df_atrp_wide_path.exists():
#     df_atrp_wide = pd.read_parquet(df_atrp_wide_path)
#     print(f"[OK] Loaded df_atrp_wide | Shape: {df_atrp_wide.shape}")
# else:
#     print(f"[ERROR] File not found: {df_atrp_wide_path}")

# if df_trp_wide_path.exists():
#     df_trp_wide = pd.read_parquet(df_trp_wide_path)
#     print(f"[OK] Loaded df_trp_wide  | Shape: {df_trp_wide.shape}")
# else:
#     print(f"[ERROR] File not found: {df_trp_wide_path}")

[ERROR] File not found: C:\Users\ping\Files_win10\python\py311\stocks\data\features_df.parquet
[ERROR] File not found: C:\Users\ping\Files_win10\python\py311\stocks\data\macro_df.parquet


In [69]:
print("🚀 Unstacking Wide Matrices...")

# 1. Unstack directly.
# (Since df_ohlcv is already a perfect grid of master_dates x tickers,
# unstacking automatically creates perfectly aligned wide dataframes).
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0)
df_trp_wide = features_df["TRP"].unstack(level=0)

print("Applying terminal NaN replacements for model inputs...")

# 2. Terminal Fills ONLY.
# Note: 0-to-NaN conversion and bounded ffills were ALREADY applied upstream to df_ohlcv.
# Applying ffill here would incorrectly double your max_data_gap_ffill limit!
df_close_wide = df_close_wide.fillna(config.nan_price_replacement)
df_atrp_wide = df_atrp_wide.fillna(0.0)
df_trp_wide = df_trp_wide.fillna(0.0)

print("All wide dataframes unstacked and prepared for downstream modeling.")
print(
    f"[OK] Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

🚀 Unstacking Wide Matrices...
Applying terminal NaN replacements for model inputs...
All wide dataframes unstacked and prepared for downstream modeling.
[OK] Pipeline Complete. Tickers: 1576, Days: 8423


In [70]:
# # Define Processed File Paths
# features_df_path = processed_dir / "features_df.parquet"
# macro_df_path = processed_dir / "macro_df.parquet"
# df_close_wide_path = processed_dir / "df_close_wide.parquet"
# df_atrp_wide_path = processed_dir / "df_atrp_wide.parquet"
# df_trp_wide_path = processed_dir / "df_trp_wide.parquet"

# # Load Processed Files
# if features_df_path.exists():
#     features_df = pd.read_parquet(features_df_path)
#     print(f"[OK] Loaded features_df  | Shape: {features_df.shape}")
# else:
#     print(f"[ERROR] File not found: {features_df_path}")

# if macro_df_path.exists():
#     macro_df = pd.read_parquet(macro_df_path)
#     print(f"[OK] Loaded macro_df     | Shape: {macro_df.shape}")
# else:
#     print(f"[ERROR] File not found: {macro_df_path}")

# if df_close_wide_path.exists():
#     df_close_wide = pd.read_parquet(df_close_wide_path)
#     print(f"[OK] Loaded df_close_wide| Shape: {df_close_wide.shape}")
# else:
#     print(f"[ERROR] File not found: {df_close_wide_path}")

# if df_atrp_wide_path.exists():
#     df_atrp_wide = pd.read_parquet(df_atrp_wide_path)
#     print(f"[OK] Loaded df_atrp_wide | Shape: {df_atrp_wide.shape}")
# else:
#     print(f"[ERROR] File not found: {df_atrp_wide_path}")

# if df_trp_wide_path.exists():
#     df_trp_wide = pd.read_parquet(df_trp_wide_path)
#     print(f"[OK] Loaded df_trp_wide  | Shape: {df_trp_wide.shape}")
# else:
#     print(f"[ERROR] File not found: {df_trp_wide_path}")

 # Section 3: In-Memory Structural Previews

 Outputs head and tail previews of the core dataframes.

In [71]:
print("\n" + "=" * 50)
print("DATAFRAME STRUCTURAL PREVIEWS (HEAD & TAIL)")
print("=" * 50)

# 1. Preview wide matrix prices
if "df_close_wide" in locals():
    print("\n=== df_close_wide (Prices) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(df_close_wide.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(df_close_wide.tail(3))

# 2. Preview multi-index features
if "features_df" in locals():
    print("\n=== features_df (Multi-Index Features) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(features_df.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(features_df.tail(3))

# 3. Preview macro data
if "macro_df" in locals():
    print("\n=== macro_df (Macro Data) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(macro_df.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(macro_df.tail(3))


DATAFRAME STRUCTURAL PREVIEWS (HEAD & TAIL)

=== df_close_wide (Prices) ===
--- HEAD (First 3 Rows) ---


Ticker,091160.KS,2330.TW,A,AA,AAL,AAON,AAPL,ABBV,ABEV,ABNB,ABT,ABVX,ACGL,ACI,ACM,ACN,ACWI,ACWX,ADBE,ADC,ADI,ADM,ADP,ADSK,ADT,AEE,AEG,AEIS,AEM,AEP,AER,AES,AFG,AFL,AFRM,AG,AGCO,AGG,AGI,AGNC,AHR,AIG,AIQ,AIRR,AIT,AIZ,AJG,AKAM,AKRE,ALAB,ALB,ALC,ALGN,ALL,ALLE,ALLY,ALNY,ALSN,ALV,AM,AMAT,AMCR,AMD,AME,AMG,AMGN,AMH,AMKR,AMLP,AMP,AMRZ,AMT,AMTM,AMX,AMZN,AN,ANET,AON,AOS,APA,APD,APG,APH,APLD,APO,APP,APPF,APTV,AR,ARCC,ARE,ARES,ARGX,ARKK,ARM,ARMK,ARWR,AS,ASML,ASND,ASR,ASTS,ASX,ATI,ATO,ATR,AU,AUR,AVAV,AVB,AVDE,AVDV,AVEM,AVGO,AVLV,AVTR,AVUS,AVUV,AVY,AWI,AWK,AXIA,AXON,AXP,AXS,AXSM,AXTA,AYI,AZN,AZO,B,BA,BABA,BAC,BAH,BAI,BALL,BAM,BAP,BAX,BBAX,BBCA,BBD,BBEU,BBIN,BBIO,BBJP,BBUS,BBVA,BBY,BCE,BCH,BCS,BDX,BE,BEKE,BEN,BEP,BEPC,BETA,BF-A,BF-B,BG,BHP,BIDU,BIIB,BIL,BILI,BINC,BIO,BIP,BIRK,BIV,BJ,BK,BKLC,BKLN,BKNG,BKR,BLD,BLDR,BLK,BLSH,BLV,BMNR,BMO,BMRN,BMY,BN,BND,BNDX,BNO,BNS,BNT,BNTX,BOKF,BOND,BOXX,BP,BPOP,BR,BRK-A,BRK-B,BRKR,BRO,BROS,BRX,BSAC,BSBR,BSCQ,BSCR,BSOL,BSV,BSX,BSY,BTC,BTI,BUD,BUFR,BURL,BVN,BWA,BWXT,BX,BXP,BYD,BZ,C,CACI,CAE,CAG,CAH,CAI,CARR,CART,CASY,CAT,CB,CBOE,CBRE,CBSH,CCEP,CCI,CCJ,CCK,CCL,CDE,CDNS,CDP,CDW,CEF,CEG,CELH,CF,CFG,CFR,CG,CGBL,CGCP,CGDV,CGGO,CGGR,CGMU,CGUS,CGXU,CHD,CHDN,CHKP,CHRW,CHT,CHTR,CHWY,CHYM,CI,CIB,CIBR,CIEN,CIFR,CIGI,CINF,CL,CLF,CLH,CLS,CLX,CM,CMC,CMCSA,CME,CMG,CMI,CMS,CNA,CNC,CNH,CNI,CNM,CNP,CNQ,COF,COHR,COIN,COKE,COLB,COO,COP,COPX,COR,CORT,COST,COWZ,CP,CPAY,CPB,CPNG,CPRT,CPT,CQP,CR,CRBG,CRCL,CRDO,CRH,CRK,CRL,CRM,CRS,CRWD,CRWV,CSCO,CSGP,CSL,CSX,CTAS,CTRE,CTSH,CTVA,CUBE,CVE,CVNA,CVS,CVX,CW,CWB,CWEN,CX,CYTK,D,DAL,DASH,DB,DBEF,DBX,DCI,DD,DDOG,DDS,DE,DECK,DELL,DEO,DFAC,DFAE,DFAI,DFAS,DFAT,DFAU,DFAX,DFCF,DFEM,DFIC,DFIS,DFIV,DFLV,DFSD,DFSV,DFUS,DFUV,DG,DGRO,DGRW,DGX,DHI,DHR,DIA,DIHP,DINO,DIS,DIVO,DKNG,DKS,DLN,DLR,DLTR,DOC,DOCS,DOCU,DON,DOV,DOW,DOX,DPZ,DRI,DRS,DSGX,DSI,DT,DTE,DTM,DUHP,DUK,DUOL,DVA,DVN,DVY,DXCM,DXJ,DY,DYNF,E,EA,EAGG,EBAY,EC,ECL,ED,EDU,EDV,EEM,EFA,EFAV,EFG,EFV,EFX,EG,EGO,EGP,EHC,EIX,EL,ELAN,ELS,ELV,EMA,EMB,EMBJ,EME,EMLC,EMN,EMR,EMXC,ENB,ENSG,ENTG,EOG,EPAM,EPD,EQH,EQIX,EQNR,EQR,EQT,EQX,ERIC,ERIE,ES,ESAB,ESGD,ESGE,ESGU,ESGV,ESLT,ESS,ESTC,ET,ETHA,ETHB,ETN,ETR,EUFN,EVR,EVRG,EVTR,EW,EWBC,EWJ,EWT,EWY,EWZ,EXC,EXE,EXEL,EXLS,EXP,EXPD,EXPE,EXR,EZU,F,FAF,FANG,FAST,FBCG,FBND,FBTC,FCFS,FCNCA,FCX,FDL,FDN,FDS,FDVV,FDX,FE,FELC,FELG,FENI,FER,FERG,FEZ,FFIV,FHN,FICO,FIG,FIGR,FIS,FISV,FITB,FIVE,FIX,FLEX,FLOT,FLR,FLS,FLUT,FMDE,FMS,FMX,FN,FND,FNDA,FNDE,FNDF,FNDX,FNF,FNV,FOX,FOXA,FPE,FR,FRHC,FRMI,FROG,FRT,FSEC,FSLR,FSS,FSV,FTAI,FTCS,FTEC,FTI,FTNT,FTS,FTSM,FTV,FUTU,FVD,FWONA,FWONK,FXI,G,GAP,GBIL,GBTC,GD,GDDY,GDS,GDX,GDXJ,GE,GEHC,GEN,GEV,GFI,GFL,GFS,GGAL,GGG,GH,GIB,GIL,GILD,GIS,GL,GLBE,GLD,GLDM,GLPI,GLW,GM,GMAB,GME,GMED,GNRC,GOOG,GOOGL,GOVT,GPC,GPN,GRAB,GRID,GRMN,GS,GSAT,GSIE,GSK,GSLC,GTLB,GTLS,GUNR,GWRE,GWW,H,HAL,HALO,HAS,HBAN,HBM,HCA,HD,HDB,HDV,HEFA,HEI,HEI-A,HESM,HIG,HII,HIMS,HL,HLI,HLN,HLNE,HLT,HMC,HMY,HON,HOOD,HPE,HPQ,HQY,HRL,HSBC,HSIC,HST,HSY,HTHT,HUBB,HUBS,HUM,HWM,HYG,IAG,IAGG,IAU,IAUM,IBB,IBIT,IBKR,IBM,IBN,IBP,ICE,ICL,ICLR,ICSH,IDA,IDCC,IDEV,IDV,IDXX,IEF,IEFA,IEI,IEMG,IESC,IEUR,IEX,IFF,IGF,IGIB,IGM,IGSB,IGV,IHG,IHI,IJH,IJJ,IJK,IJR,IJS,IJT,ILMN,IMO,INCY,INDA,INFY,ING,INGR,INSM,INTC,INTU,INVH,IONQ,IONS,IOO,IOT,IP,IQLT,IQV,IR,IREN,IRM,ISRG,ISTB,IT,ITA,ITOT,ITT,ITUB,ITW,IUSB,IUSG,IUSV,IVE,IVV,IVW,IVZ,IWB,IWD,IWF,IWM,IWN,IWO,IWP,IWR,IWS,IWV,IWY,IX,IXJ,IXN,IXUS,IYF,IYR,IYW,J,JAAA,JAVA,JAZZ,JBHT,JBL,JBND,JBS,JBTM,JCI,JCPB,JD,JEF,JEPI,JEPQ,JGLO,JGRO,JHG,JHMM,JHX,JIRE,JKHY,JLL,JMBS,JMST,JMTG,JMUB,JNJ,JNK,JOBY,JPIE,JPM,JPST,JQUA,JXN,KB,KBWB,KDP,KEP,KEY,KEYS,KGC,KHC,KIM,KKR,KLAC,KLAR,KMB,KMI,KNSL,KNX,KO,KR,KRE,KRMN,KRYS,KSPI,KT,KTOS,KVUE,KVYO,KWEB,KYMR,L,LAD,LAMR,LBRDA,LBRDK,LDOS,LECO,LEN,LEVI,LFUS,LH,LHX,LI,LII,LIN,LINE,LITE,LKQ,LLY,LLYVA,LLYVK,LMBS,LMT,LNC,LNG,LNT,LOGI,LOW,LPLA,LQD,LRCX,LSCC,LTM,LULU,LUMN,LUV,LVS,LW,LYB,LYFT,LYG,LYV,MA,MAA,MAGS,MANH,MAR,MAS,MBB,MBLY,MCD,MCHI,MCHP,MCK,MCO,MDB,MDGL,MDLZ,MDT,MDY,MEDP,MELI,MET,META,MFC,MFG,MGA,MGC,MGK,MGM,MGV,MHK,MICC,MIDD,MINT,MKC,MKL,MKSI,MKTX,MLI,MLM,MMM,MMYT,MNDY,MNST,MO,MOAT,MOD,MOG-A,MOH,M


--- TAIL (Last 3 Rows) ---


Ticker,091160.KS,2330.TW,A,AA,AAL,AAON,AAPL,ABBV,ABEV,ABNB,ABT,ABVX,ACGL,ACI,ACM,ACN,ACWI,ACWX,ADBE,ADC,ADI,ADM,ADP,ADSK,ADT,AEE,AEG,AEIS,AEM,AEP,AER,AES,AFG,AFL,AFRM,AG,AGCO,AGG,AGI,AGNC,AHR,AIG,AIQ,AIRR,AIT,AIZ,AJG,AKAM,AKRE,ALAB,ALB,ALC,ALGN,ALL,ALLE,ALLY,ALNY,ALSN,ALV,AM,AMAT,AMCR,AMD,AME,AMG,AMGN,AMH,AMKR,AMLP,AMP,AMRZ,AMT,AMTM,AMX,AMZN,AN,ANET,AON,AOS,APA,APD,APG,APH,APLD,APO,APP,APPF,APTV,AR,ARCC,ARE,ARES,ARGX,ARKK,ARM,ARMK,ARWR,AS,ASML,ASND,ASR,ASTS,ASX,ATI,ATO,ATR,AU,AUR,AVAV,AVB,AVDE,AVDV,AVEM,AVGO,AVLV,AVTR,AVUS,AVUV,AVY,AWI,AWK,AXIA,AXON,AXP,AXS,AXSM,AXTA,AYI,AZN,AZO,B,BA,BABA,BAC,BAH,BAI,BALL,BAM,BAP,BAX,BBAX,BBCA,BBD,BBEU,BBIN,BBIO,BBJP,BBUS,BBVA,BBY,BCE,BCH,BCS,BDX,BE,BEKE,BEN,BEP,BEPC,BETA,BF-A,BF-B,BG,BHP,BIDU,BIIB,BIL,BILI,BINC,BIO,BIP,BIRK,BIV,BJ,BK,BKLC,BKLN,BKNG,BKR,BLD,BLDR,BLK,BLSH,BLV,BMNR,BMO,BMRN,BMY,BN,BND,BNDX,BNO,BNS,BNT,BNTX,BOKF,BOND,BOXX,BP,BPOP,BR,BRK-A,BRK-B,BRKR,BRO,BROS,BRX,BSAC,BSBR,BSCQ,BSCR,BSOL,BSV,BSX,BSY,BTC,BTI,BUD,BUFR,BURL,BVN,BWA,BWXT,BX,BXP,BYD,BZ,C,CACI,CAE,CAG,CAH,CAI,CARR,CART,CASY,CAT,CB,CBOE,CBRE,CBSH,CCEP,CCI,CCJ,CCK,CCL,CDE,CDNS,CDP,CDW,CEF,CEG,CELH,CF,CFG,CFR,CG,CGBL,CGCP,CGDV,CGGO,CGGR,CGMU,CGUS,CGXU,CHD,CHDN,CHKP,CHRW,CHT,CHTR,CHWY,CHYM,CI,CIB,CIBR,CIEN,CIFR,CIGI,CINF,CL,CLF,CLH,CLS,CLX,CM,CMC,CMCSA,CME,CMG,CMI,CMS,CNA,CNC,CNH,CNI,CNM,CNP,CNQ,COF,COHR,COIN,COKE,COLB,COO,COP,COPX,COR,CORT,COST,COWZ,CP,CPAY,CPB,CPNG,CPRT,CPT,CQP,CR,CRBG,CRCL,CRDO,CRH,CRK,CRL,CRM,CRS,CRWD,CRWV,CSCO,CSGP,CSL,CSX,CTAS,CTRE,CTSH,CTVA,CUBE,CVE,CVNA,CVS,CVX,CW,CWB,CWEN,CX,CYTK,D,DAL,DASH,DB,DBEF,DBX,DCI,DD,DDOG,DDS,DE,DECK,DELL,DEO,DFAC,DFAE,DFAI,DFAS,DFAT,DFAU,DFAX,DFCF,DFEM,DFIC,DFIS,DFIV,DFLV,DFSD,DFSV,DFUS,DFUV,DG,DGRO,DGRW,DGX,DHI,DHR,DIA,DIHP,DINO,DIS,DIVO,DKNG,DKS,DLN,DLR,DLTR,DOC,DOCS,DOCU,DON,DOV,DOW,DOX,DPZ,DRI,DRS,DSGX,DSI,DT,DTE,DTM,DUHP,DUK,DUOL,DVA,DVN,DVY,DXCM,DXJ,DY,DYNF,E,EA,EAGG,EBAY,EC,ECL,ED,EDU,EDV,EEM,EFA,EFAV,EFG,EFV,EFX,EG,EGO,EGP,EHC,EIX,EL,ELAN,ELS,ELV,EMA,EMB,EMBJ,EME,EMLC,EMN,EMR,EMXC,ENB,ENSG,ENTG,EOG,EPAM,EPD,EQH,EQIX,EQNR,EQR,EQT,EQX,ERIC,ERIE,ES,ESAB,ESGD,ESGE,ESGU,ESGV,ESLT,ESS,ESTC,ET,ETHA,ETHB,ETN,ETR,EUFN,EVR,EVRG,EVTR,EW,EWBC,EWJ,EWT,EWY,EWZ,EXC,EXE,EXEL,EXLS,EXP,EXPD,EXPE,EXR,EZU,F,FAF,FANG,FAST,FBCG,FBND,FBTC,FCFS,FCNCA,FCX,FDL,FDN,FDS,FDVV,FDX,FE,FELC,FELG,FENI,FER,FERG,FEZ,FFIV,FHN,FICO,FIG,FIGR,FIS,FISV,FITB,FIVE,FIX,FLEX,FLOT,FLR,FLS,FLUT,FMDE,FMS,FMX,FN,FND,FNDA,FNDE,FNDF,FNDX,FNF,FNV,FOX,FOXA,FPE,FR,FRHC,FRMI,FROG,FRT,FSEC,FSLR,FSS,FSV,FTAI,FTCS,FTEC,FTI,FTNT,FTS,FTSM,FTV,FUTU,FVD,FWONA,FWONK,FXI,G,GAP,GBIL,GBTC,GD,GDDY,GDS,GDX,GDXJ,GE,GEHC,GEN,GEV,GFI,GFL,GFS,GGAL,GGG,GH,GIB,GIL,GILD,GIS,GL,GLBE,GLD,GLDM,GLPI,GLW,GM,GMAB,GME,GMED,GNRC,GOOG,GOOGL,GOVT,GPC,GPN,GRAB,GRID,GRMN,GS,GSAT,GSIE,GSK,GSLC,GTLB,GTLS,GUNR,GWRE,GWW,H,HAL,HALO,HAS,HBAN,HBM,HCA,HD,HDB,HDV,HEFA,HEI,HEI-A,HESM,HIG,HII,HIMS,HL,HLI,HLN,HLNE,HLT,HMC,HMY,HON,HOOD,HPE,HPQ,HQY,HRL,HSBC,HSIC,HST,HSY,HTHT,HUBB,HUBS,HUM,HWM,HYG,IAG,IAGG,IAU,IAUM,IBB,IBIT,IBKR,IBM,IBN,IBP,ICE,ICL,ICLR,ICSH,IDA,IDCC,IDEV,IDV,IDXX,IEF,IEFA,IEI,IEMG,IESC,IEUR,IEX,IFF,IGF,IGIB,IGM,IGSB,IGV,IHG,IHI,IJH,IJJ,IJK,IJR,IJS,IJT,ILMN,IMO,INCY,INDA,INFY,ING,INGR,INSM,INTC,INTU,INVH,IONQ,IONS,IOO,IOT,IP,IQLT,IQV,IR,IREN,IRM,ISRG,ISTB,IT,ITA,ITOT,ITT,ITUB,ITW,IUSB,IUSG,IUSV,IVE,IVV,IVW,IVZ,IWB,IWD,IWF,IWM,IWN,IWO,IWP,IWR,IWS,IWV,IWY,IX,IXJ,IXN,IXUS,IYF,IYR,IYW,J,JAAA,JAVA,JAZZ,JBHT,JBL,JBND,JBS,JBTM,JCI,JCPB,JD,JEF,JEPI,JEPQ,JGLO,JGRO,JHG,JHMM,JHX,JIRE,JKHY,JLL,JMBS,JMST,JMTG,JMUB,JNJ,JNK,JOBY,JPIE,JPM,JPST,JQUA,JXN,KB,KBWB,KDP,KEP,KEY,KEYS,KGC,KHC,KIM,KKR,KLAC,KLAR,KMB,KMI,KNSL,KNX,KO,KR,KRE,KRMN,KRYS,KSPI,KT,KTOS,KVUE,KVYO,KWEB,KYMR,L,LAD,LAMR,LBRDA,LBRDK,LDOS,LECO,LEN,LEVI,LFUS,LH,LHX,LI,LII,LIN,LINE,LITE,LKQ,LLY,LLYVA,LLYVK,LMBS,LMT,LNC,LNG,LNT,LOGI,LOW,LPLA,LQD,LRCX,LSCC,LTM,LULU,LUMN,LUV,LVS,LW,LYB,LYFT,LYG,LYV,MA,MAA,MAGS,MANH,MAR,MAS,MBB,MBLY,MCD,MCHI,MCHP,MCK,MCO,MDB,MDGL,MDLZ,MDT,MDY,MEDP,MELI,MET,META,MFC,MFG,MGA,MGC,MGK,MGM,MGV,MHK,MICC,MIDD,MINT,MKC,MKL,MKSI,MKTX,MLI,MLM,MMM,MMYT,MNDY,MNST,MO,MOAT,MOD,MOG-A,MOH,M


=== features_df (Multi-Index Features) ===
--- HEAD (First 3 Rows) ---


...

ATR      ATRP       TRP       RSI    Mom_21  Consistency     IR_63   Beta_63     DD_21  AutoCorr_15    Ret_1d  Range_Pos_20  Slope_P_5  Slope_V_5  Slope_P_5_Z  Slope_V_5_Z  Convexity  RollingStalePct  RollMedDollarVol  RollingSameVolCount
Ticker Date                                                                                                                                                                                                                                                           
ZWS    2026-07-15  1.236545  0.025961  0.017636 -0.082696 -0.015706          0.6 -0.083843  1.101098 -0.071359    -0.121936  0.002526      0.204263   0.003982   0.255324     0.453519     0.541443   0.005059              0.0      4.021030e+07                  0.0
       2026-07-16  1.261078  0.026066  0.032658  0.003042 -0.009216          0.8 -0.030634  1.139130 -0.056736     0.103883  0.015746      0.337478   0.005785   0.433177     0.628594     0.827152   0.002432              0.0      4.021030e+07                  0.0
       2026-07-17  1.247429  0.026135  0.022418 -0.065991 -0.037119          0.6 -0.016506  1.149861 -0.069409    -0.208103 -0.013435      0.222025   0.005195   0.354094     0.546427     0.668873   0.001214              0.0      4.045033e+07                  0.0


=== macro_df (Macro Data) ===
--- HEAD (First 3 Rows) ---


,Mkt_Ret,Mkt_Ret_Z,Macro_Trend,Macro_Trend_Z,High_Yield_Spread_Z,Yield_Curve_10Y2Y_Z,Macro_Trend_Vel_Z,Macro_Trend_Mom,Macro_Vix_Z,Macro_Vix_Ratio,Mkt_Vol_63d_Z
Date,,,,,,,,,,,
1997-01-02,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1997-01-03,0.014353,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1997-01-06,-0.008739,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0



--- TAIL (Last 3 Rows) ---


,Mkt_Ret,Mkt_Ret_Z,Macro_Trend,Macro_Trend_Z,High_Yield_Spread_Z,Yield_Curve_10Y2Y_Z,Macro_Trend_Vel_Z,Macro_Trend_Mom,Macro_Vix_Z,Macro_Vix_Ratio,Mkt_Vol_63d_Z
Date,,,,,,,,,,,
2026-07-15,0.003964,0.314838,0.089911,0.210533,-1.272054,-1.469186,0.351333,0.351333,-1.298617,0.828662,0.353342
2026-07-16,-0.005419,-0.810075,0.083267,0.003627,-1.261809,-1.456801,-1.243963,-1.243963,-0.543182,0.857949,0.361127
2026-07-17,-0.009897,-1.316496,0.071888,-0.347941,-1.251313,-1.443830,-1.528960,-1.528960,0.864151,0.913827,0.463496


 # Section 4: Engine Initialization & Stage 1 Analysis

In [72]:
# Instantiate Master Engine
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("Master AlphaEngine Ready.")

Master AlphaEngine Ready.


In [73]:
df_ohlcv.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 8308398 entries, ('091160.KS', Timestamp('2007-01-29 00:00:00')) to ('ZWS', Timestamp('2026-07-17 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     int64  
dtypes: float64(4), int64(1)
memory usage: 349.0+ MB


In [74]:
import pandas as pd

target_date = pd.Timestamp("2025-09-01")

# Use level=1 for the date level (since level names aren't 'timestamp')
dates_in_index = df_ohlcv.index.get_level_values(1).normalize().unique()
print(f"2025-09-01 in index: {target_date in dates_in_index}")

# Check if date exists (one-liner)
any_match = (df_ohlcv.index.get_level_values(1).normalize() == target_date).any()
print(f"Any match: {any_match}")

# If you want to see which tickers have data on that date:
if any_match:
    mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
    tickers_on_date = df_ohlcv.index.get_level_values(0)[mask].unique()
    print(f"Tickers on 2025-09-01: {tickers_on_date}")
    print(f"Count: {len(tickers_on_date)}")

# Extract all data for that specific date
if any_match:
    # Using boolean mask (works regardless of index sorting)
    mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
    df_target = df_ohlcv[mask]
    print(df_target.head())

2025-09-01 in index: False
Any match: False


In [75]:
df_ohlcv.index

MultiIndex([('091160.KS', '2007-01-29'),
            ('091160.KS', '2007-01-30'),
            ('091160.KS', '2007-01-31'),
            ('091160.KS', '2007-02-01'),
            ('091160.KS', '2007-02-02'),
            ('091160.KS', '2007-02-05'),
            ('091160.KS', '2007-02-06'),
            ('091160.KS', '2007-02-07'),
            ('091160.KS', '2007-02-08'),
            ('091160.KS', '2007-02-09'),
            ...
            (      'ZWS', '2026-07-06'),
            (      'ZWS', '2026-07-07'),
            (      'ZWS', '2026-07-08'),
            (      'ZWS', '2026-07-09'),
            (      'ZWS', '2026-07-10'),
            (      'ZWS', '2026-07-13'),
            (      'ZWS', '2026-07-14'),
            (      'ZWS', '2026-07-15'),
            (      'ZWS', '2026-07-16'),
            (      'ZWS', '2026-07-17')],
           names=['Ticker', 'Date'], length=8308398)

In [76]:
# Configuration for Interactive UI (Stage 1 Setup)
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=189,
    holding_period=5,
    metric="Sharpe (TRP)",
    benchmark_ticker=config.benchmark_ticker,
    rank_start=1,
    rank_end=200,
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

In [77]:
# Execute Headless Agent Simulation
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df.style.format("{:.4f}"))
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
----------------------------------------------------------------------
Timeline: [2025-07-16] -> Decision: 2026-04-16 -> Entry: 2026-04-17 -> End: 2026-04-24
Selected Tickers (200):
SGOV, SHV, BIL, MINT, USFR, BOXX, PULS, JPST, GTLS, SNDK
JAAA, RVMD, CIEN, TER, EA, LITE, ASX, VGSH, WDC, GLW
ROIV, SHY, ROST, EWY, WBD, VALE, MU, FTI, ERIC, RIO
VTEB, BE, KEYS, NOK, STX, FIX, UTHR, VCSH, IGSB, MUB
CHRW, EMXC, LRCX, GOOGL, GOOG, HSBC, INTC, TEVA, JNJ, TD
CAT, EWZ, ALB, MKSI, CMI, BNS, BSV, COPX, RY, SU
PBR, EFV, ASML, GSK, EWT, FDX, ONTO, BP, AA, IAG
CVE, SLV, HL, KGC, ITUB, COHR, BHP, AU, JAZZ, IAUM
AMAT, GLDM, SGOL, IAU, APA, B, AZN, XBI, SCCO, EMB
PSLV, EQX, SOXX, SCHF, GLD, VEA, BG, SPDW, NEM, IEMG
VEU, IONS, GDX, EEM, VXUS, PHYS, LSCC, HAL, IXUS, ETR
EWJ, MTZ, OPEN, SMH, FNDX, USO, SOXL, KLAC, GDXJ, MOD
NBIS, FTAI, SIL, MEDP, VLO, MPWR, SPIB, VRT, SHEL, AEM
MBB, TTMI, JBHT, BKR, CNQ, IDEV, DD, GFI, ATI, WELL
FE, JNK, EQIX, NVS, FNV, MTSI, VTR, NVT, EIX,

,Full,Lookback,Holding
Metric,,,
Group Gain,0.6829,0.6567,0.0033
Benchmark Gain,0.1427,0.1254,0.0053
== Gain Delta,0.5402,0.5313,-0.0020
Group Sharpe,4.0042,3.9418,1.4141
Benchmark Sharpe,1.5411,1.4013,2.3401
== Sharpe Delta,2.4631,2.5405,-0.9259
Group Sharpe (ATRP),0.1104,0.1101,0.0237
Benchmark Sharpe (ATRP),0.0723,0.0662,0.0879
== Sharpe (ATRP) Delta,0.0381,0.0439,-0.0643



Target Reward Signal: 0.003314


 # Section 5: Stage 2 - Cascade Filter Execution

In [78]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# LAUNCH STAGE 2 (Cascade on the Stage 1 subset)
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [79]:
# Parse and map results from Analyzer 2
result = SU.map_analyzer(analyzer2)

[SEARCH] ANALYZER SIMULATION MAP
[  0] [DATA] audit_pack (EngineOutput)
[  1]   [PLOT] portfolio_series (shape=(196,))
[  2]   [PLOT] benchmark_series (shape=(196,))
[  3]   [CALC] normalized_plot_data (shape=(196, 100))
[  4]   [FILE] tickers (len=100)
[  5]     [DOC] index_0 (str)
[  6]     [DOC] index_1 (str)
[  7]     [DOC] index_2 (str)
[  8]     [DOC] index_3 (str)
[  9]     [DOC] index_4 (str)
[ 10]     [DOC] index_5 (str)
[ 11]     [DOC] index_6 (str)
[ 12]     [DOC] index_7 (str)
[ 13]     [DOC] index_8 (str)
[ 14]     [DOC] index_9 (str)
[ 15]     [DOC] index_10 (str)
[ 16]     [DOC] index_11 (str)
[ 17]     [DOC] index_12 (str)
[ 18]     [DOC] index_13 (str)
[ 19]     [DOC] index_14 (str)
[ 20]     [DOC] index_15 (str)
[ 21]     [DOC] index_16 (str)
[ 22]     [DOC] index_17 (str)
[ 23]     [DOC] index_18 (str)
[ 24]     [DOC] index_19 (str)
[ 25]     [DOC] index_20 (str)
[ 26]     [DOC] index_21 (str)
[ 27]     [DOC] index_22 (str)
[ 28]     [DOC] index_23 (str)
[ 29]     [D

In [80]:
# Test different methods to extract resulting target tickers
_tickers = SU.fetch(4, result)
print(f"_tickers (by index): {_tickers}\n")

_tickers = SU.fetch("tickers", result)
print(f"_tickers (by key): {_tickers}\n")

print(f'result[4]["obj"]: {result[4]["obj"]}')

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM',
 'IEMG']

_tickers (by index): ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC', 'RIO', 'VTEB', 'BE', 'KEYS', 'NOK', 'STX', 'FIX', 'UTHR', 'VCSH', 'IGSB', 'MUB', 'CHRW', 'EMXC', 'LRCX', 'GOOGL', 'GOOG', 'HSBC', 'INTC', 'TEVA', 'JNJ', 'TD', 'CAT', 'EWZ', 'ALB', 'MKSI', 'CMI', 'BNS', 'BSV', 'COPX', 'RY', 'SU', 'PBR', 'EFV', 'ASML', 'GSK', 'EWT', 'FDX', 'ONTO', 'BP', 'AA', 'IAG', 'CVE', 'SLV', 'HL', 'KGC', 'ITUB', 'COHR', 'BHP', 'AU', 'JAZZ', 'IAUM', 'AMAT', 'GLDM', 'SGOL', 'IAU', 'APA', 'B', 'AZN', 'XBI', 'SCCO', 'EMB', 'PSLV', 'EQX', 'SOXX', 'SCHF', 'GLD', 'VEA', 'BG', 'SPDW', 'NEM', 'IEMG']

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM',
 'IEMG']

_tickers (by key): ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC', 'RIO', 'VTEB', 'BE', 'KEYS', 'NOK', 'STX', 'FIX', 'UTHR', 'VCSH', 'IGSB', 'MUB', 'CHRW', 'EMXC', 'LRCX', 'GOOGL', 'GOOG', 'HSBC', 'INTC', 'TEVA', 'JNJ', 'TD', 'CAT', 'EWZ', 'ALB', 'MKSI', 'CMI', 'BNS', 'BSV', 'COPX', 'RY', 'SU', 'PBR', 'EFV', 'ASML', 'GSK', 'EWT', 'FDX', 'ONTO', 'BP', 'AA', 'IAG', 'CVE', 'SLV', 'HL', 'KGC', 'ITUB', 'COHR', 'BHP', 'AU', 'JAZZ', 'IAUM', 'AMAT', 'GLDM', 'SGOL', 'IAU', 'APA', 'B', 'AZN', 'XBI', 'SCCO', 'EMB', 'PSLV', 'EQX', 'SOXX', 'SCHF', 'GLD', 'VEA', 'BG', 'SPDW', 'NEM', 'IEMG']

result[4]["obj"]: ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC', 'RIO', 

 # Section 6: Verification, Exports & Auditing

In [81]:
# Export analytical trace logs to Excel
SU.export_audit_to_excel(
    audit_pack=analyzer2.last_run, filename="Audit_Verification_Report.xlsx"
)

[FILE] [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx
[DONE] Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/Audit_Verification_Report.xlsx')

In [82]:
# Export last-run stacked OHLCV ticker records
SU.export_last_run_tickers_data_to_csv(
    analyzer=analyzer2,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename="last_run_tickers_stacked.csv",
)

Creating combined dictionary for 101 ticker(s)
Date range: 2025-07-16 00:00:00 to 2026-04-24 00:00:00
Data retrieved for 101 ticker(s) from 2025-07-16 00:00:00 to 2026-04-24 00:00:00
Total rows: 19796
Date range in data: 2025-07-16 00:00:00 to 2026-04-24 00:00:00
  SGOV: 196 rows
  SHV: 196 rows
  BIL: 196 rows
  MINT: 196 rows
  USFR: 196 rows
  BOXX: 196 rows
  PULS: 196 rows
  JPST: 196 rows
  GTLS: 196 rows
  SNDK: 196 rows
  JAAA: 196 rows
  RVMD: 196 rows
  CIEN: 196 rows
  TER: 196 rows
  EA: 196 rows
  LITE: 196 rows
  ASX: 196 rows
  VGSH: 196 rows
  WDC: 196 rows
  GLW: 196 rows
  ROIV: 196 rows
  SHY: 196 rows
  ROST: 196 rows
  EWY: 196 rows
  WBD: 196 rows
  VALE: 196 rows
  MU: 196 rows
  FTI: 196 rows
  ERIC: 196 rows
  RIO: 196 rows
  VTEB: 196 rows
  BE: 196 rows
  KEYS: 196 rows
  NOK: 196 rows
  STX: 196 rows
  FIX: 196 rows
  UTHR: 196 rows
  VCSH: 196 rows
  IGSB: 196 rows
  MUB: 196 rows
  CHRW: 196 rows
  EMXC: 196 rows
  LRCX: 196 rows
  GOOGL: 196 rows
  GOOG: 

WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/last_run_tickers_stacked.csv')

 # Section 7: Slicing Tests & Dictionary Parsing

In [83]:
# Verify captured ticker names
_tickers

['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM',
 'IEMG']

In [84]:
# Test various pandas multi-index slicing configurations
test_tickers = ["NVDA", "GOOG"]
idx = pd.IndexSlice
columns = ["Adj Close", "Volume"]

# Slice 1: All dates, all columns
display(df_ohlcv.loc[idx[test_tickers]].copy().head(3))

# Slice 2: specific start date, all columns
display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], :].head(3))

# Slice 3: specific start date, selected columns
display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], columns].head(3))

# Slice 4: specific start/end range, all columns
display(
    df_ohlcv.loc[
        idx[test_tickers, pd.Timestamp("2026-04-01") : pd.Timestamp("2026-04-10")], :
    ].head(3)
)

Adj Open  Adj High   Adj Low  Adj Close      Volume
Ticker Date                                                           
NVDA   1999-01-22  0.040063  0.044713  0.035532   0.037559  2964547212
       1999-01-25  0.040540  0.041970  0.037559   0.041494   557464453
       1999-01-26  0.041970  0.042805  0.037678   0.038274   374788019

Adj Open  Adj High  Adj Low  Adj Close     Volume
Ticker Date                                                         
NVDA   2026-04-01   175.795   177.164  174.547    175.545  168327949
       2026-04-02   171.980   177.283  171.170    177.183  143310037
       2026-04-06   176.954   177.583  175.555    177.433  107689668

Adj Close     Volume
Ticker Date                            
NVDA   2026-04-01    175.545  168327949
       2026-04-02    177.183  143310037
       2026-04-06    177.433  107689668

Adj Open  Adj High  Adj Low  Adj Close     Volume
Ticker Date                                                         
NVDA   2026-04-01   175.795   177.164  174.547    175.545  168327949
       2026-04-02   171.980   177.283  171.170    177.183  143310037
       2026-04-06   176.954   177.583  175.555    177.433  107689668

In [85]:
# Test custom combined dictionary extraction
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=_tickers,
    date_start="2026-05-01",
    date_end=None,
    verbose=False,
)

{'SGOV':             Adj Open  Adj High   Adj Low  Adj Close    Volume       ATR      ATRP       TRP       RSI    Mom_21  Consistency     IR_63   Beta_63  DD_21  AutoCorr_15    Ret_1d  Range_Pos_20  Slope_P_5  Slope_V_5  Slope_P_5_Z  Slope_V_5_Z     Convexity  RollingStalePct  RollMedDollarVol  RollingSameVolCount
 Date                                                                                                                                                                                                                                                                                                              
 2026-05-01   99.8174   99.8174   99.8075    99.8174  34010416  0.019380  0.000194  0.000379  0.999997  0.003169          0.6 -0.061535 -0.003308    0.0    -0.574082  0.000379      1.000000   0.000145   0.739754     0.083808    -0.303915  6.269530e-06         0.011905      1.368667e+09                  0.0
 2026-05-04   99.8274   99.8274   99.8174    99.8274  20905678  0.01

 # Section 8: Finviz Ingestion & Filtering

In [86]:
# Load latest updated Finviz dataset dynamically
def load_latest_finviz_parquet(data_dir: Path) -> pd.DataFrame:
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"
    files = list(data_dir.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {data_dir.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Load the file using the unified data_dir path
df_finviz = load_latest_finviz_parquet(GLOBAL_DATA_DIR)
print(
    f"df_finviz raw records shape: {df_finviz.shape if df_finviz is not None else None}"
)

DEBUG: Loading file: 2026-07-17_df_finviz_merged_stocks_etfs.parquet
df_finviz raw records shape: (1426, 139)


In [87]:
display(df_finviz.head())
display(df_finviz.tail())

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
NVDA,1,NVIDIA Corp,"DJIA, NDX, S&P 500",Technology,Semiconductors,USA,NASD,"Technology, Semiconductors",5016420.0,1,5016420.0,31.75,NaN,0.34,19.79,25.69,62.26,42.13,8.07,3.33,0.31,0.28,6/4/2026,0.82,6.53,2.08,NaN,NaN,NaN,NaN,NaN,85.23,213.42,109.57,70.68,253490.0,159610.0,6.52,3.43,24220.0,23270.0,96.07,3.85,-0.33,70.44,0.00,1.33,1.99,310.13,82.97,114.29,77.17,3.44,2.85,0.06,0.07,74.15,64.02,62.97,-4.244570,-2.13,-1.61,2.59,11.31,20.95,11.15,2.22,7.20,3.473395,3.13,3.31,2.78,-1.17,7.64,-12.37,9.21,-12.37,26.34,164.07 - 236.54,-12.37,621769.95,52.24,May 20/a,1/22/1999,Yes,Yes,42000.0,-0.12,2.10,1.23,156.20,0.70,108685563,313.74,203.28,207.54,208.65,204.01,207.29,1.97,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-278.986352,-15.861674,0.000000,-0.372315,-0.682930,0.939110,2.321686,4.178263,1.434780,2.460691,4.368984,1.467117,-1.458425,-1.954923,0.796620,0.142354,0.214060,1.022596,0.550344,0.825261,1.092388,0.544999,0.799288,1.092006
AAPL,2,Apple Inc,"DJIA, NDX, S&P 500",Technology,Consumer Electronics,USA,NASD,"Technology, Consumer Electronics",4813630.0,2,4813630.0,39.65,NaN,2.62,10.66,45.14,70.26,37.26,7.26,4.66,0.33,1.05,5/11/2026,13.66,8.27,1.89,NaN,NaN,NaN,NaN,NaN,16.60,22.04,28.91,12.76,451440.0,122580.0,3.30,1.58,14690.0,14670.0,99.88,0.12,-2.21,67.21,0.00,0.96,2.55,140.53,34.91,141.47,67.76,1.07,1.02,0.70,0.80,47.86,32.64,27.15,5.996316,4.09,9.98,20.03,28.26,54.25,20.55,1.07,8.21,2.505034,2.64,2.79,6.24,7.73,19.21,-2.16,19.72,-2.16,62.65,201.50 - 334.99,-2.16,515084.78,64.54,Jul 30/a,12/12/1980,Yes,Yes,166000.0,1.43,-1.06,2.04,55.10,0.75,41338917,321.28,326.59,323.13,329.60,322.22,327.74,0.35,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,13.006772,9918.843399,884.640802,9.611452,51.186814,7.448932,7.760268,32.155268,4.721830,10.679806,53.271708,7.567258,1.806564,2.680799,1.363610,3.080708,4.852769,1.708333,2.099500,3.195737,1.431509,1.839780,2.947064,1.387544
GOOGL,3,Alphabet Inc,"DJIA, NDX, S&P 500",Communication Services,Internet Content & Information,USA,NASD,"Communication Services, Internet Content & Inf...",4222970.0,3,4222970.0,26.48,NaN,1.37,9.98,8.79,33.29,65.54,39.51,10.43,0.25,0.85,6/8/2026,7.68,13.11,2.88,NaN,NaN,NaN,NaN,NaN,22.34,81.96,46.22,17.77,423170.0,160210.0,90.47,2.73,12170.0,5830.0,47.91,52.09,-0.04,39.25,0.00,1.47,2.70,85.86,27.17,38.88,28.04,1.92,1.92,0.19,0.20,60.43,33.63,37.86,-3.543712,-3.44,-5.67,2.88,5.20,82.61,10.91,1.24,11.29,3.252196,3.38,3.00,-2.20,-5.83,7.57,-15.04,5.13,-15.04,86.49,186.15 - 408.61,-15.04,14356.60,43.10,Jul 22/a,8/19/2004,Yes,Yes,

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
SBIL,394,Simplify Government Money Market ETF,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, Bonds - Money...",5030.0,1554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.55,3.56,6/25/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.07,0.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.039916,0.07,0.01,0.02,0.01,0.19,0.19,-0.00,0.05,0.049860,0.02,0.03,0.12,0.10,0.10,-0.19,0.27,-0.43,0.27,100.01 - 100.71,-0.43,0.27,66.31,-,7/15/2025,No,Yes,NaN,0.00,0.01,NaN,0.25224,0.32,80043,NaN,100.27,100.28,100.29,100.27,100.28,0.01,Bonds - Money Market,Bonds,0.15,166.0,5030.0,NaN,2.22,NaN,4.78,615.99,13.96,3.87,NaN,NaN,"U.S., fixed-income, debt, bonds, government-bonds",-0.679437,-1.281318,0.885851,-6.631362,-8.063849,0.373569,1.090253,1.572784,1.183270,-1.368435,-1.758833,0.808439,-2.277987,-2.840040,0.703918,-1.390021,-1.759697,0.795634,-1.983742,-2.565138,0.731820,-0.913346,-1.278601,0.863335
EAGG,396,iShares ESG Aware U.S. Aggregate Bond ETF,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, Bonds - Broad...",4990.0,1556,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.04,1.89,7/1/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.07,1.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.234442,-0.23,-1.14,-2.11,-2.34,-0.76,-2.17,0.26,0.15,0.320444,0.21,0.20,-0.72,-0.80,-1.95,-1.72,0.22,-3.68,0.22,46.71 - 48.60,-18.15,6.22,37.73,-,10/23/2018,Yes,Yes,NaN,-0.13,-0.09,NaN,0.40631,0.37,151407,NaN,46.91,46.87,46.87,46.79,46.81,-0.21,Bonds - Broad Market,Bonds,0.10,5142.0,4990.0,NaN,3.62,NaN,7.75,688.12,16.00,3.93,3.58,-0.24,"U.S., fixed-income, corporate-bonds, debt-secu...",-4.186487,-6.098475,0.456705,11.517052,55.175694,7.951484,-4.142448,-4.823654,0.514101,-5.476661,-5.850667,0.394584,-1.197108,-1.593790,0.823879,-1.432123,-1.874108,0.796103,-1.027892,-1.353351,0.848708,-0.059391,-0.083883,0.990489
SJNK,397,State Street SPDR Bloomberg Short Term High Yi...,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, Bonds - Corpo...",4920.0,1557,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.03,1.75,7/1/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.55,1.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.040145,-0.08,-0.60,-1.35,-2.20,-2.08,-1.70,0.29,0.05,0.200884,0.13,0.15,-0.24,-0.37,-1.17,-1.03,0.12,-2.96,0.69,24.72 - 25.65,-20.17,16.20,42.00,-,3/15/2012,Yes,Yes,NaN,-0.16,0.04,NaN,2.47000,0.62,1535859,NaN,24.92,24.

In [88]:
# Set test target tickers
target_tickers = ["GLW", "NOK", "DELL", "NVDA", "GOOG"]

In [89]:
# Clean index extraction using ast exception parsing if tickers are missing
try:
    result = df_finviz.loc[target_tickers]
except KeyError as e:
    error_msg = str(e)
    missing_str = error_msg.replace(" not in index", "").strip('"').strip("'")
    missing_tickers = ast.literal_eval(missing_str)

    print(f"Missing tickers: {missing_tickers}")

    # Strip out the missing symbols and retry slicing
    target_tickers_in_finviz = [t for t in target_tickers if t not in missing_tickers]
    result = df_finviz.loc[target_tickers_in_finviz].copy()
    result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

print(f"result shape: {result.shape}")

result shape: (5, 139)


In [90]:
# Display filtered Finviz information and schema details
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, GLW to GOOG
Columns: 139 entries, No. to Omega 250d
dtypes: float64(120), int64(3), object(16)
memory usage: 5.5+ KB


In [91]:
# Check specific target column (Market Capitalization)
result["MktCap AUM, M"]

GLW      139780.0
NOK       59450.0
DELL     262000.0
NVDA    5016420.0
GOOG    4218720.0
Name: MktCap AUM, M, dtype: float64

In [92]:
# Sort records descendingly by Market Capitalization
result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

In [93]:
# View final sorted DataFrame
print(result)

      No.                Company               Index                  Sector                        Industry  Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M    P/E  Fwd P/E   PEG    P/S    P/B    P/C  P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %    EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %   ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W Low

In [94]:
# Export processed dataframe output directly to trash file
result.to_csv(output_dir / "_trash.csv")